In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_2_CVAE import CVAE
from e_1_run_cvae import train_chunk
#from e_1_run_cvae_time_check import train_chunk
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # hes or bs
barr_type = 'van' # van or barr
opt_type = 'call' # call or put
chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# training

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 5e-6
l2          = 3
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 92 # 1 chunk train : 3m, 92+9:283m / 5+5:40m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 100
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt"

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 5e-06
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=194->286 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -5.2598 | KL: 4.5263 | Total: -0.7335
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -5.2616 | KL: 4.5395 | Total: -0.7221
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -5.2636 | KL: 4.5461 | Total: -0.7174
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -5.2676 | KL: 4.5372 | Total: -0.7305
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -5.2720 | KL: 4.5390 | Total: -0.7330
Chu

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 2
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_8_128_4096_None_1e-05_1_[15, 24, 78]_chunk92.pt | 완료 chunks=92
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=92->97 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=2 | bn_chunks=None | warmup_chunks=None
Chunk step    93 | epoch    1 chunk  93/97 | file_idx  43 | BN off    | beta_eff: 1.0000 | Recon: -5.2197 | KL: 3.9578 | Total: -1.2619
Chunk step    94 | epoch    1 chunk  94/97 | file_idx  48 | BN off    | beta_eff: 1.0000 | Recon: -5.2174 | KL: 3.9550 | Total: -1.2624
Validation @ chunk    94 | Recon: -5.2186 | KL: 3.9615 | Total: -1.2571 | KL_dim: [2.1e-05, 4.8e-05, 2.639203, 4e-05, 5.4e-05, 5.7e-05, 1.321981, 6.1e-05]
Chunk step    95 | epoch    1 chunk  95/97 | file_idx  85 | BN off    | beta_eff: 1.0000 | Recon: -5.2170 | KL: 3.9590 | Total: -1.2579
Chunk step    96 | epoch    1 chunk  96/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -5.2216 | KL: 3.9577 | Tota

In [ ]:
# CVAE training settings
lr          = 1e-5 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-6
l2          = 3
lr3         = 1e-6
l3          = 4
num_chunks  = 92
val_every_chunks = 100
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk286.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk97.pt | 완료 chunks=97
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=97->189 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step    98 | epoch    2 chunk   1/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.2290 | KL: 4.5183 | Total: -0.7107
Chunk step    99 | epoch    2 chunk   2/97 | file_idx  14 | BN off    | beta_eff: 1.0000 | Recon: -5.2339 | KL: 4.5101 | Total: -0.7239
Chunk step   100 | epoch    2 chunk   3/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -5.2353 | KL: 4.4998 | Total: -0.7356
Validation @ chunk   100 | Recon: -5.2351 | KL: 4.5074 | Total: -0.7277 | KL_dim: [2.967619, 1.539766]
Chunk step   101 | epoch    2 chunk   4/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.2385 | KL: 4.5016 | Total: -0.7369
Chunk step   102 | epoch    2 chunk   5

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 5
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk189.pt | 완료 chunks=189
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=189->194 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=5 | bn_chunks=None | warmup_chunks=None
Chunk step   190 | epoch    2 chunk  93/97 | file_idx  31 | BN off    | beta_eff: 1.0000 | Recon: -5.2431 | KL: 4.5261 | Total: -0.7170
Validation @ chunk   190 | Recon: -5.2484 | KL: 4.5159 | Total: -0.7325 | KL_dim: [2.952767, 1.563079]
Chunk step   191 | epoch    2 chunk  94/97 | file_idx  33 | BN off    | beta_eff: 1.0000 | Recon: -5.2493 | KL: 4.5066 | Total: -0.7427
Chunk step   192 | epoch    2 chunk  95/97 | file_idx  32 | BN off    | beta_eff: 1.0000 | Recon: -5.2499 | KL: 4.5095 | Total: -0.7404
Chunk step   193 | epoch    2 chunk  96/97 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -5.2487 | KL: 4.5034 | Total: -0.7453
Chunk step   194 | epoch    2 chunk  97

In [5]:
# CVAE training settings
lr          = 1e-5 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-5
l2          = 3
lr3         = 1e-6
l3          = 4
num_chunks  = 92
val_every_chunks = 100
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk286.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=194->286 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -5.2477 | KL: 4.5154 | Total: -0.7323
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -5.2428 | KL: 4.5220 | Total: -0.7207
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -5.2416 | KL: 4.5251 | Total: -0.7165
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -5.2454 | KL: 4.5160 | Total: -0.7294
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -5.2479 | KL: 4.5164 | Total: -0.7316
Chu

In [6]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 5
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk286.pt | 완료 chunks=286
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=286->291 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=5 | bn_chunks=None | warmup_chunks=None
Chunk step   287 | epoch    3 chunk  93/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.2679 | KL: 4.5487 | Total: -0.7193
Chunk step   288 | epoch    3 chunk  94/97 | file_idx  73 | BN off    | beta_eff: 1.0000 | Recon: -5.2736 | KL: 4.5408 | Total: -0.7329
Chunk step   289 | epoch    3 chunk  95/97 | file_idx  56 | BN off    | beta_eff: 1.0000 | Recon: -5.2717 | KL: 4.5422 | Total: -0.7294
Chunk step   290 | epoch    3 chunk  96/97 | file_idx   5 | BN off    | beta_eff: 1.0000 | Recon: -5.2706 | KL: 4.5458 | Total: -0.7248
Validation @ chunk   290 | Recon: -5.2731 | KL: 4.5385 | Total: -0.7346 | KL_dim: [2.95186, 1.586698]
Chunk step   291 | epoch    3 chunk  97/

In [7]:
# CVAE training settings
lr          = 1e-5 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-5
l2          = 3
lr3         = 1e-6
l3          = 4
num_chunks  = 92
val_every_chunks = 100
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk383.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk291.pt | 완료 chunks=291
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=291->383 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step   292 | epoch    4 chunk   1/97 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -5.2745 | KL: 4.5407 | Total: -0.7338
Chunk step   293 | epoch    4 chunk   2/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.2692 | KL: 4.5513 | Total: -0.7180
Chunk step   294 | epoch    4 chunk   3/97 | file_idx  12 | BN off    | beta_eff: 1.0000 | Recon: -5.2712 | KL: 4.5367 | Total: -0.7345
Chunk step   295 | epoch    4 chunk   4/97 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -5.2742 | KL: 4.5435 | Total: -0.7307
Chunk step   296 | epoch    4 chunk   5/97 | file_idx  94 | BN off    | beta_eff: 1.0000 | Recon: -5.2753 | KL: 4.5443 | Total: -0.7309
Chu

In [8]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 5
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk383.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk383.pt | 완료 chunks=383
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=383->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=5 | bn_chunks=None | warmup_chunks=None
Chunk step   384 | epoch    4 chunk  93/97 | file_idx   4 | BN off    | beta_eff: 1.0000 | Recon: -5.3011 | KL: 4.5654 | Total: -0.7356
Chunk step   385 | epoch    4 chunk  94/97 | file_idx  69 | BN off    | beta_eff: 1.0000 | Recon: -5.2979 | KL: 4.5735 | Total: -0.7244
Validation @ chunk   385 | Recon: -5.2985 | KL: 4.5651 | Total: -0.7334 | KL_dim: [2.957077, 1.608063]
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.3066 | KL: 4.5617 | Total: -0.7449
Chunk step   387 | epoch    4 chunk  96/97 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -5.3064 | KL: 4.5636 | Total: -0.7428
Chunk step   388 | epoch    4 chunk  97

In [9]:
# CVAE training settings
dim_z       = 8
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 1
lr2         = 1e-5
l2          = 2
lr3         = 1e-6
l3          = 4
num_chunks  = 92
val_every_chunks = 100
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk189.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_8_128_4096_None_0.0001_1_[15, 24, 78]_chunk97.pt | 완료 chunks=97
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=97->189 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step    98 | epoch    2 chunk   1/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -4.9886 | KL: 4.2686 | Total: -0.7200
Chunk step    99 | epoch    2 chunk   2/97 | file_idx  14 | BN off    | beta_eff: 1.0000 | Recon: -5.0651 | KL: 4.3318 | Total: -0.7333
Chunk step   100 | epoch    2 chunk   3/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -5.0943 | KL: 4.3491 | Total: -0.7452
Validation @ chunk   100 | Recon: -5.1014 | KL: 4.3644 | Total: -0.7370 | KL_dim: [8e-06, 6e-06, 6e-06, 5e-06, 1.518104, 6e-06, 2.846254, 4e-06]
Chunk step   101 | epoch    2 chunk   4/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.1143 | KL: 4.3677 | Total: -0.7

In [10]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 3
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk189.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk194.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_8_128_4096_None_1-0.0001_2-1e-05_1_[15, 24, 78]_chunk189.pt | 완료 chunks=189
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=189->194 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=3 | bn_chunks=None | warmup_chunks=None
Chunk step   190 | epoch    2 chunk  93/97 | file_idx  31 | BN off    | beta_eff: 1.0000 | Recon: -5.2502 | KL: 4.5268 | Total: -0.7233
Chunk step   191 | epoch    2 chunk  94/97 | file_idx  33 | BN off    | beta_eff: 1.0000 | Recon: -5.2558 | KL: 4.5064 | Total: -0.7494
Chunk step   192 | epoch    2 chunk  95/97 | file_idx  32 | BN off    | beta_eff: 1.0000 | Recon: -5.2571 | KL: 4.5104 | Total: -0.7467
Validation @ chunk   192 | Recon: -5.2552 | KL: 4.5162 | Total: -0.7390 | KL_dim: [7e-06, 3e-06, 1e-05, 2e-05, 1.594322, 3e-06, 2.92187, 5e-06]
Chunk step   193 | epoch    2 chunk  96/97 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -5.2575 | KL: 4.5055 | To

# BN = 5

In [2]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = 5 # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-5
l2          = 3
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 92
validation_chunk_idxs = [15,24,78]
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_0.0001_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=194->286 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=10 | bn_chunks=5 | warmup_chunks=None
Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN frozen | beta_eff: 1.0000 | Recon: -5.3391 | KL: 4.5933 | Total: -0.7458
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN frozen | beta_eff: 1.0000 | Recon: -5.4229 | KL: 4.6885 | Total: -0.7345
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN frozen | beta_eff: 1.0000 | Recon: -5.4530 | KL: 4.7225 | Total: -0.7305
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN frozen | beta_eff: 1.0000 | Recon: -5.4805 | KL: 4.7367 | Total: -0.7437
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN frozen | beta_eff: 1.0000 | Recon: -5.4988 | KL: 4.7528 | Total: -0.7460
Chunk ste

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_3-0.0001_4-1e-06_1_[15, 24, 78]_chunk383.pt | 완료 chunks=383
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=383->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=5 | warmup_chunks=None
Chunk step   384 | epoch    4 chunk  93/97 | file_idx   4 | BN frozen | beta_eff: 1.0000 | Recon: -5.2304 | KL: 4.4934 | Total: -0.7370
Validation @ chunk   384 | Recon: -5.2282 | KL: 4.4857 | Total: -0.7425 | KL_dim: [1.436707, 3.048991]
Chunk step   385 | epoch    4 chunk  94/97 | file_idx  69 | BN frozen | beta_eff: 1.0000 | Recon: -5.2268 | KL: 4.5010 | Total: -0.7258
Validation @ chunk   385 | Recon: -5.2273 | KL: 4.4858 | Total: -0.7415 | KL_dim: [1.42515, 3.060651]
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN frozen | beta_eff: 1.0000 | Recon: -5.2385 | KL: 4.4922 | Total: -0.7463
Validation @ chunk   386 | Recon: -5.2095 | KL: 4.4750 | Total: -0.7345 | KL_dim: [1.42636

In [ ]:
# CVAE training settings
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-6
l2          = 3
num_chunks  = 92
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

# X,M norm

In [ ]:
# CVAE training settings
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 0.001 # 0.001,0.0003, 0.0005
beta        = 0.9
warmup_chunks = None # None or num
num_chunks  = 70
validation_chunk_idxs = [15,24,78]
val_every_chunks = 3
resume_path = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk30.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk100.pt"

In [6]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_norm(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    x_mean=bs_stats["x_mean"],
    x_std=bs_stats["x_std"],
    m_mean=bs_stats["m_mean"],
    m_std=bs_stats["m_std"],
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae_xm_norm/bs/cvae_bs_8_128_4096_None_0.001_0.9_None_chunk30.pt | 완료 chunks=30
학습 시작 | 이번 실행 chunks=70 | 진행 chunks=30->100 | files/epoch=100 | bn_chunks=None | warmup_chunks=None
Chunk step    31 | epoch    1 chunk  31/100 | file_idx  30 | BN off    | beta_eff: 0.9000 | Recon: -4.9477 | KL: 5.3827 | Total: -0.1033
Chunk step    32 | epoch    1 chunk  32/100 | file_idx  29 | BN off    | beta_eff: 0.9000 | Recon: -4.8912 | KL: 6.1971 | Total: 0.6862
Chunk step    33 | epoch    1 chunk  33/100 | file_idx  79 | BN off    | beta_eff: 0.9000 | Recon: -4.9538 | KL: 6.2571 | Total: 0.6776
Chunk step    34 | epoch    1 chunk  34/100 | file_idx  44 | BN off    | beta_eff: 0.9000 | Recon: -5.0059 | KL: 5.4327 | Total: -0.1164
Chunk step    35 | epoch    1 chunk  35/100 | file_idx  71 | BN off    | beta_eff: 0.9000 | Recon: -4.9941 | KL: 5.4282 | Total: -0.1088
Chunk step    36 | epoch    1 chunk  36/100 | file_idx  66 | BN off    | beta_eff: 0.9000 | Rec